# [TEMPLATE] <Tên Phương án> — Colab Runner

> **Đây là notebook MẪU, chưa chạy được ngay** — mọi chỗ đánh dấu `# TODO(...)` cần được điền trước
> khi dùng thật. Xem quy trình đầy đủ tại `Folder_Base/HUONG_DAN_XAY_DUNG_FOLDER.md`. Mọi phần hạ
> tầng còn lại (clone luôn lấy code mới nhất, tự dò entry point khi repo bị lồng thư mục, tải dữ liệu
> Google Drive tự nhận diện link file/thư mục **hoặc mount Drive cá nhân**, chạy huấn luyện qua
> `subprocess` với log tuyệt đối và kiểm tra exit code, đọc kết quả có kiểm tra phòng vệ, xuất bảng +
> biểu đồ biến thiên theo epoch + PDF tự động, và 1 cell Optuna tuỳ chọn để dò siêu tham số) đã được
> kiểm chứng kỹ ở các phương án trước — **giữ nguyên, không viết lại**.

**Cách dùng khi đã điền xong TODO:** chạy lần lượt từng cell từ trên xuống. Cell 1 là nơi duy nhất cần
chỉnh mỗi lần dùng (link GitHub + Google Drive + số epoch). Nhớ bật GPU:
`Runtime > Change runtime type > Hardware accelerator > GPU`. Cell 8 (Optuna) ở cuối là **tuỳ chọn**,
chỉ chạy nếu muốn tự động dò siêu tham số.


## 1. Cell cấu hình đầu vào

Đây là **cell duy nhất người dùng cuối cần chỉnh sửa** mỗi lần chạy. Khi điền template, thêm mọi
hyperparameter riêng của phương án mới vào đúng đây (theo mẫu `SIGMA_MIN` đã comment sẵn bên dưới).
`DATASET_NAME` dùng để đặt tên và ghi chú trong file PDF kết quả ở Cell 7 — nếu thuật toán không có
khái niệm nhiều dataset, cứ để 1 giá trị cố định (ví dụ tên dataset duy nhất mà repo hỗ trợ).

`USE_GDRIVE_MOUNT` chọn cách lấy dữ liệu ở Cell 4: `False` (mặc định) tải công khai qua `GDRIVE_LINK`
bằng `gdown` (đơn giản, nhưng dễ bị Google chặn "quota exceeded" nếu file được nhiều người tải);
`True` mount Drive **cá nhân** của người đang chạy Colab và copy trực tiếp — không bị giới hạn quota,
nhưng cần người dùng tự có sẵn dữ liệu trong Drive của họ (xem chú thích ở Cell 4).


In [ ]:
# ============================================================
# CELL 1 — CẤU HÌNH ĐẦU VÀO (chỗ DUY NHẤT người dùng cần chỉnh sửa)
# ============================================================

GITHUB_REPO_URL = ""  # <-- dán URL repo GitHub CỦA BẠN (đã push bản fork đã patch — xem README)
GDRIVE_LINK = ""       # <-- dán link Google Drive chứa dữ liệu (link file .zip hoặc link thư mục)
USE_GDRIVE_MOUNT = False  # True: mount Drive CÁ NHÂN của bạn ở Cell 4 (tránh lỗi quota gdown khi tải công khai) — xem chú thích Cell 4
DATASET_NAME = ""      # TODO: tên dataset dùng để chạy (vd "tiktok" | "baby" | "sports") — dùng để đặt tên file PDF kết quả
NUM_EPOCHS = 50         # <-- chỉnh số epoch mong muốn ở đây

# TODO(điền khi tạo phương án mới): thêm hyperparameter RIÊNG của phương án vào đây, ví dụ:
# SIGMA_MIN = 1e-3  # [Ví dụ tham khảo từ Phương án 1] tham số riêng của phương án

# TODO (tuỳ chọn): dừng sớm nếu chỉ số Test không cải thiện sau N epoch — CHỈ có tác dụng nếu bản fork
# đã patch thêm `--patience` vào Params.py + logic dừng trong Main.py's run() (xem mẫu patch ở
# Folder_Base/HUONG_DAN_XAY_DUNG_FOLDER.md, mục "Early stopping (tuỳ chọn)"). Nếu bản fork CHƯA patch
# hỗ trợ này mà vẫn truyền --patience ở Cell 6, script gốc sẽ báo lỗi "unrecognized arguments".
# PATIENCE = 5  # 0 = tắt

assert GITHUB_REPO_URL.strip() != "", "Hãy dán URL repo GitHub của bạn vào GITHUB_REPO_URL ở trên trước khi chạy tiếp."
assert GDRIVE_LINK.strip() != "", "Hãy dán link Google Drive vào biến GDRIVE_LINK ở trên trước khi chạy tiếp."
assert DATASET_NAME.strip() != "", "Hãy điền tên dataset vào DATASET_NAME ở trên trước khi chạy tiếp."
print(f"Cấu hình: repo={GITHUB_REPO_URL}, dataset={DATASET_NAME}, epochs={NUM_EPOCHS}, use_gdrive_mount={USE_GDRIVE_MOUNT}")


## 2. Cell setup môi trường

In [ ]:
# ============================================================
# CELL 2 — SETUP MÔI TRƯỜNG
# ============================================================
import torch

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Chưa bật GPU cho Colab. Vào Runtime > Change runtime type > Hardware accelerator > GPU, "
    "rồi Runtime > Restart session và chạy lại từ Cell 1."
)

# TODO(điền khi tạo phương án mới): thêm thư viện phụ trợ mà repo gốc cần (kiểm tra các dòng import
# ở đầu mỗi file .py gốc) — gdown/tabulate luôn cần cho hạ tầng notebook, giữ nguyên 2 cái này.
!pip install -q gdown tabulate  # TODO: thêm thư viện khác nếu cần, ví dụ: setproctitle, ...

print("Đã cài đặt xong các thư viện cần thiết.")

## 3. Cell clone code

**Hạ tầng đã kiểm chứng — không cần sửa**, trừ 2 biến `REPO_DIR` và `MAIN_SCRIPT_NAME` ở đầu cell.
Cell này luôn xoá bản clone cũ rồi clone lại từ đầu mỗi lần chạy (tránh lỗi dùng nhầm code cũ khi chạy
lại trong cùng phiên Colab), và tự dò tìm entry point script trong toàn bộ cây thư mục vừa clone,
tự động điều chỉnh đường dẫn nếu repo bị lồng thêm cấp thư mục.

In [ ]:
# ============================================================
# CELL 3 — CLONE CODE TỪ REPO GITHUB CỦA BẠN
# ============================================================
import glob
import os
import shutil

REPO_DIR = "REPO_DIR_LOCAL"  # TODO: đổi tên thư mục local (tuỳ ý đặt, không bắt buộc trùng tên repo)
MAIN_SCRIPT_NAME = "Main.py"  # TODO: đổi thành tên file entry point THẬT của thuật toán (xem README gốc)

# Luôn xoá bản clone cũ (nếu có) rồi clone lại từ đầu — đảm bảo LUÔN lấy đúng code mới nhất trên
# GitHub. Nếu không làm vậy, chạy lại Cell 3 trong cùng 1 phiên Colab (mà không Restart runtime) sẽ bị
# bỏ qua bước clone và dùng nhầm code CŨ đã tải trước đó.
if os.path.isdir(REPO_DIR):
    print(f"Xoá bản clone cũ '{REPO_DIR}' để lấy code MỚI NHẤT từ GitHub...")
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 {GITHUB_REPO_URL} {REPO_DIR}

assert os.path.isdir(REPO_DIR), (
    f"Không tìm thấy thư mục '{REPO_DIR}' sau khi clone — kiểm tra lại GITHUB_REPO_URL ở Cell 1 "
    "(repo phải công khai, hoặc bạn đã đăng nhập git trên Colab nếu là repo private)."
)

# Tự dò entry point trong toàn bộ cây thư mục vừa clone, phòng trường hợp repo bị lồng thêm 1 cấp
# thư mục (ví dụ push nhầm cả folder cha, hoặc upload qua giao diện web GitHub thay vì git push).
main_script_candidates = glob.glob(os.path.join(REPO_DIR, "**", MAIN_SCRIPT_NAME), recursive=True)
assert main_script_candidates, (
    f"Clone thành công nhưng KHÔNG tìm thấy '{MAIN_SCRIPT_NAME}' ở đâu trong '{REPO_DIR}'.\n"
    f"Nội dung hiện có: {sorted(os.listdir(REPO_DIR))}\n"
    "Kiểm tra lại: GITHUB_REPO_URL đúng repo chưa, MAIN_SCRIPT_NAME đúng tên file chưa, và repo có "
    "đúng nội dung bản fork đã patch hay không."
)
main_script_candidates.sort(key=lambda p: p.count(os.sep))
actual_dir = os.path.dirname(main_script_candidates[0])

if actual_dir != REPO_DIR:
    print(
        f"⚠ {MAIN_SCRIPT_NAME} không nằm trực tiếp trong '{REPO_DIR}' mà nằm trong '{actual_dir}' "
        "— có thể repo bị lồng thêm 1 cấp thư mục. Tự động dùng đường dẫn này cho các bước sau."
    )
    REPO_DIR = actual_dir

print(f"\nREPO_DIR đang dùng: {REPO_DIR}")
print(sorted(os.listdir(REPO_DIR)))

## 4. Cell tải dữ liệu

**Hạ tầng đã kiểm chứng — không cần sửa**, trừ 2 biến `TARGET_DATA_DIR` và `REQUIRED_DATA_FILES` ở
đầu cell. Hỗ trợ **2 chế độ** (chọn bằng `USE_GDRIVE_MOUNT` ở Cell 1):

- **`False` (mặc định) — tải công khai qua `gdown`:** nhận cả link file `.zip` lẫn link thư mục, tự
  giải nén (kể cả zip lồng bên trong), tự dò tìm thư mục chứa file dữ liệu đầu tiên trong
  `REQUIRED_DATA_FILES` bất kể cấu trúc bên trong Drive ra sao, ưu tiên thư mục con trùng tên
  `DATASET_NAME` nếu tìm thấy nhiều ứng viên (tránh lẫn dữ liệu giữa nhiều dataset share chung 1 Drive).
- **`True` — mount Drive cá nhân:** dùng khi link Drive công khai bị Google chặn tải do quá nhiều lượt
  tải (`quota exceeded`) — yêu cầu người dùng tự thêm lối tắt (shortcut) thư mục dữ liệu vào **Drive của
  họ**, mặc định tìm tại `MyDrive/DiffMM_Data/<DATASET_NAME>/` (đổi `DRIVE_DATA_PATH` trong cell nếu
  muốn đường dẫn khác). Tự nhận diện khi đang dry-run cục bộ (không phải Colab thật) để bỏ qua bước
  mount, tránh lỗi khi kiểm thử ngoài Colab.


In [ ]:
# ============================================================
# CELL 4 — CHUẨN BỊ DỮ LIỆU HUẤN LUYỆN
# ============================================================
import glob
import os
import shutil
import zipfile

TARGET_DATA_DIR = os.path.join(REPO_DIR, "TODO_thu_muc_du_lieu")  # TODO: đúng đường dẫn mà entry point script yêu cầu
REQUIRED_DATA_FILES = []  # TODO: liệt kê tên các file bắt buộc, vd ["trnMat.pkl", "tstMat.pkl", "image_feat.npy"]

assert REQUIRED_DATA_FILES, "TODO: điền REQUIRED_DATA_FILES ở trên trước khi chạy cell này."

os.makedirs(TARGET_DATA_DIR, exist_ok=True)

if USE_GDRIVE_MOUNT:
    # Che do 2: mount Drive CA NHAN cua nguoi dang chay Colab (tranh loi quota khi ca nhieu nguoi cung
    # tai 1 link cong khai). Yeu cau: nguoi dung da tu them loi tat (shortcut) thu muc du lieu vao
    # "Drive cua toi" (MyDrive) truoc khi chay, dat dung ten nhu DRIVE_DATA_PATH ben duoi.
    is_dryrun = not os.path.exists("/content")  # tu nhan dien khong chay trong Colab that (vd dry-run cuc bo)
    if is_dryrun:
        print("[Dry-run cục bộ] Bỏ qua bước mount Google Drive (không phải môi trường Colab thật)...")
        for f in REQUIRED_DATA_FILES:
            with open(os.path.join(TARGET_DATA_DIR, f), "w") as out:
                out.write("fake data")
    else:
        print("Đang kết nối tới Google Drive cá nhân của bạn...")
        from google.colab import drive
        drive.mount("/content/gdrive")

        DRIVE_DATA_PATH = f"/content/gdrive/MyDrive/DiffMM_Data/{DATASET_NAME}/"  # TODO: đổi nếu bạn đặt dữ liệu ở đường dẫn khác trong Drive
        print(f"Sao chép dữ liệu từ: {DRIVE_DATA_PATH} sang {TARGET_DATA_DIR}")
        for fname in REQUIRED_DATA_FILES:
            src_file = os.path.join(DRIVE_DATA_PATH, fname)
            dst_file = os.path.join(TARGET_DATA_DIR, fname)
            assert os.path.exists(src_file), (
                f"Không tìm thấy file '{fname}' tại đường dẫn: {src_file}\n"
                "Kiểm tra lại: (1) bạn đã thêm lối tắt thư mục dữ liệu vào 'Drive của tôi' (MyDrive) "
                f"chưa, (2) tên thư mục/file trong {DATASET_NAME} có đúng DRIVE_DATA_PATH ở trên không."
            )
            shutil.copy2(src_file, dst_file)
            print(f"✓ Đã copy: {fname}")
else:
    # Che do 1 (mac dinh): tai cong khai qua gdown, khong yeu cau nguoi dung co san du lieu trong Drive
    import gdown
    DOWNLOAD_DIR = "gdrive_download"
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)

    if "/folders/" in GDRIVE_LINK:
        print("Phát hiện link THƯ MỤC Google Drive -> tải cả thư mục...")
        gdown.download_folder(url=GDRIVE_LINK, output=DOWNLOAD_DIR, quiet=False, use_cookies=False)
    else:
        print("Phát hiện link FILE Google Drive -> tải file...")
        downloaded_path = gdown.download(
            url=GDRIVE_LINK, output=os.path.join(DOWNLOAD_DIR, "gdrive_data"), quiet=False, fuzzy=True
        )
        assert downloaded_path, "Tải dữ liệu từ Google Drive thất bại — kiểm tra lại link (phải ở chế độ chia sẻ công khai/Anyone with the link). Nếu bị lỗi 'quota exceeded', đổi USE_GDRIVE_MOUNT=True ở Cell 1 và dùng Drive cá nhân thay thế."
        if downloaded_path.lower().endswith(".zip"):
            print(f"Giải nén {downloaded_path} ...")
            with zipfile.ZipFile(downloaded_path, "r") as zf:
                zf.extractall(DOWNLOAD_DIR)

    # Tự giải nén mọi file .zip lồng bên trong (ví dụ do dữ liệu gốc đóng gói zip lồng nhau)
    for _ in range(3):
        zip_files = glob.glob(os.path.join(DOWNLOAD_DIR, "**", "*.zip"), recursive=True)
        if not zip_files:
            break
        for zpath in zip_files:
            try:
                with zipfile.ZipFile(zpath, "r") as zf:
                    zf.extractall(os.path.dirname(zpath))
                print(f"Đã giải nén: {zpath}")
                os.remove(zpath)
            except zipfile.BadZipFile:
                pass

    # Tự dò tìm thư mục chứa file dữ liệu đầu tiên trong REQUIRED_DATA_FILES, bất kể cấu trúc Drive.
    # Neu tim thay NHIEU ung vien (vi du Drive chia se chung nhieu dataset), uu tien thu muc co ten
    # trung DATASET_NAME de tranh lay nham du lieu cua dataset khac.
    marker_file = REQUIRED_DATA_FILES[0]
    candidates = glob.glob(os.path.join(DOWNLOAD_DIR, "**", marker_file), recursive=True)
    filtered_candidates = [c for c in candidates if f"/{DATASET_NAME.lower()}/" in c.replace(os.sep, "/").lower()]
    if filtered_candidates:
        candidates = filtered_candidates
    assert candidates, (
        f"Không tìm thấy '{marker_file}' trong dữ liệu tải về từ Google Drive.\n"
        "Hãy kiểm tra: (1) link đã ở chế độ chia sẻ công khai (Anyone with the link) chưa, "
        f"(2) dữ liệu có đủ các file: {REQUIRED_DATA_FILES} hay chưa."
    )
    src_dir = os.path.dirname(candidates[0])
    print("Tìm thấy dữ liệu tại:", src_dir)

    for fname in os.listdir(src_dir):
        fpath = os.path.join(src_dir, fname)
        if os.path.isfile(fpath):
            shutil.copy2(fpath, TARGET_DATA_DIR)

missing = [f for f in REQUIRED_DATA_FILES if not os.path.exists(os.path.join(TARGET_DATA_DIR, f))]
assert not missing, f"Thiếu file trong {TARGET_DATA_DIR}: {missing}. Kiểm tra lại dữ liệu trên Google Drive."

print(f"\nDữ liệu đã sẵn sàng tại: {TARGET_DATA_DIR}")
print(sorted(os.listdir(TARGET_DATA_DIR)))


## 5. Cell xác minh code đã có đúng patch

Cell này **cần viết riêng cho từng phương án** (không có hạ tầng chung, vì mỗi patch có tên
class/hàm/argument khác nhau). Mẫu tham khảo đầy đủ: xem Cell 5 trong
`phuong_an_1_OT_noise_scheduler/DiffMM_PhuongAn1_OT_Colab.ipynb` — kiểm tra sự tồn tại của
class/argument mới bằng cách đọc nội dung file .py rồi kiểm tra chuỗi đặc trưng, in ✓/✗ rõ ràng, và
`assert` toàn bộ điều kiện đều đúng trước khi cho phép chạy tiếp.

In [ ]:
# ============================================================
# CELL 5 — XÁC MINH REPO ĐÃ CLONE CÓ ĐÚNG PATCH (viết riêng cho từng phương án)
# ============================================================

# TODO: điền đường dẫn các file cần kiểm tra, ví dụ:
# model_path = os.path.join(REPO_DIR, "Model.py")
# with open(model_path, "r", encoding="utf-8") as f:
#     model_src = f.read()
#
# checks = {
#     "Model.py có class <TenClassMoi>": "class <TenClassMoi>" in model_src,
#     # TODO: thêm các điều kiện kiểm tra khác
# }
#
# for name, ok in checks.items():
#     print(("✓ " if ok else "✗ ") + name)
#
# assert all(checks.values()), (
#     "Repo vừa clone KHÔNG có đúng patch mong đợi. Kiểm tra lại GITHUB_REPO_URL ở Cell 1 có trỏ đúng "
#     "repo/branch chứa bản đã patch hay không."
# )
# print("\n--- Repo đã clone đúng là bản có patch ---")

raise NotImplementedError("TODO: điền logic xác minh patch cho phương án này rồi xoá dòng raise này.")

## 6. Cell chạy huấn luyện

**Hạ tầng đã kiểm chứng — không cần sửa cấu trúc**, chỉ điền `DATASET_HP`/`CLI_ARGS` cho đúng lệnh
chạy của thuật toán mới (lấy từ README gốc). Chạy bằng `subprocess` (không dùng
`!cd ... && ... | tee ...`) để log có đường dẫn tuyệt đối, và để **báo lỗi ngay tại đây** (kèm exit
code + toàn bộ output) nếu script chính thoát với lỗi, thay vì để lỗi trôi xuống Cell 7 dưới dạng
`FileNotFoundError` khó hiểu.

Nếu đã patch thêm hỗ trợ dừng sớm (`--patience`, xem `HUONG_DAN_XAY_DUNG_FOLDER.md`) và khai báo biến
`PATIENCE` ở Cell 1, nhớ nối thêm `"--patience", str(PATIENCE)` vào `CLI_ARGS` bên dưới.


In [ ]:
# ============================================================
# CELL 6 — CHẠY HUẤN LUYỆN
# ============================================================
import subprocess

# TODO: điền hyperparameter khuyến nghị (lấy từ README gốc), ví dụ:
# DATASET_HP = {
#     "tiktok": ["--reg", "1e-4", "--ssl_reg", "1e-2"],
# }
DATASET_HP = {}  # TODO

# TODO: build danh sách args CLI cho lệnh chạy chính, ví dụ:
# CLI_ARGS = ["--epoch", str(NUM_EPOCHS), "--sigma_min", str(SIGMA_MIN)] + DATASET_HP["tiktok"]
# TODO (tuỳ chọn, chỉ nếu bản fork đã patch --patience): thêm "--patience", str(PATIENCE) vào CLI_ARGS
CLI_ARGS = []  # TODO

assert CLI_ARGS, "TODO: điền CLI_ARGS ở trên trước khi chạy cell này."

# LOG_PATH dùng đường dẫn TUYỆT ĐỐI, tính trước khi đổi thư mục làm việc cho tiến trình con, để
# Cell 7 (và cả việc tự mở file lên xem) luôn tìm đúng file log, không phụ thuộc cwd hiện tại.
LOG_PATH = os.path.abspath("train_log.txt")

cmd = ["python", MAIN_SCRIPT_NAME] + CLI_ARGS  # TODO: đổi "python" nếu script cần cách chạy khác

print("Lệnh chạy:", " ".join(cmd))
print("Thư mục làm việc:", os.path.abspath(REPO_DIR))
print("Log sẽ được ghi vào:", LOG_PATH)
print()

with open(LOG_PATH, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, universal_newlines=True,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    process.wait()

print(f"\n\n{MAIN_SCRIPT_NAME} kết thúc với exit code: {process.returncode}")
assert process.returncode == 0, (
    f"{MAIN_SCRIPT_NAME} thoát với lỗi (exit code {process.returncode}). Xem log phía trên (hoặc mở "
    f"file {LOG_PATH}) để biết chi tiết lỗi — sửa xong thì chạy lại Cell 6 trước khi sang Cell 7."
)
assert os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0, f"Không tạo được file log tại {LOG_PATH}."
print(f"Huấn luyện xong, đã ghi log đầy đủ vào: {LOG_PATH}")


## 7. Cell xuất kết quả (bảng + biểu đồ theo epoch + file PDF tự động)

**Hạ tầng đã kiểm chứng (phần kiểm tra phòng vệ + phần xuất PDF/biểu đồ) — chỉ cần điền `RESULT_REGEX`
và cách map kết quả**, y hệt trước đây. Đọc kỹ code phần in kết quả cuối cùng của script gốc (hoặc chạy
thử 1 lần) để biết chính xác định dạng dòng log chứa chỉ số tốt nhất, viết regex khớp đúng định dạng
đó — không đoán.

**Phần xuất PDF bảng kết quả không cần sửa gì** — tự động lấy tên phương án từ cột `"Phương án"` và tên
dataset từ biến `DATASET_NAME` (Cell 1) để đặt tên/tiêu đề file, dùng `matplotlib` (đã có sẵn trên
Colab, không cần cài thêm) để vẽ bảng `result_df` thành 1 trang PDF, rồi **tự động tải file PDF về máy
qua trình duyệt** (dùng `google.colab.files.download` — chỉ hoạt động khi chạy thật trên Colab).

**Phần bảng chi tiết + biểu đồ biến thiên theo epoch** (`EPOCH_REGEX`) dùng sẵn định dạng dòng log
chuẩn của các bản fork DiffMM (`Epoch %d/%d, Test: Recall = ..., NDCG = ..., Precision = ...`, xem hàm
`makePrint` trong `Main.py` gốc) — **không cần sửa nếu bản fork của bạn giữ nguyên `makePrint`**, chỉ
cần sửa nếu định dạng dòng log khác. Nếu không khớp được dòng nào, cell tự bỏ qua phần biểu đồ (in
thông báo rõ ràng) chứ không báo lỗi — an toàn để luôn để bật.


In [ ]:
# ============================================================
# CELL 7 — XUẤT KẾT QUẢ (bảng + biểu đồ theo epoch + tự động xuất & tải file PDF)
# ============================================================
import re

import pandas as pd

assert "LOG_PATH" in globals() and os.path.exists(LOG_PATH), (
    "Không tìm thấy file log huấn luyện (biến LOG_PATH chưa có hoặc file không tồn tại). "
    "Nguyên nhân thường gặp nhất: Cell 6 chưa được chạy trong phiên này, hoặc phiên Colab đã bị "
    "reset/ngắt kết nối (mất hết file tạm) giữa lúc chạy Cell 6 và Cell 7. "
    "=> Hãy chạy lại Cell 6, đợi huấn luyện xong hẳn, rồi chạy lại Cell 7 này."
)

with open(LOG_PATH, "r", encoding="utf-8") as f:
    log_text = f.read()

# TODO: điền regex khớp đúng dòng in kết quả tốt nhất của script gốc, ví dụ (mẫu từ Phương án 1):
# RESULT_REGEX = r"Best epoch\s*:\s*(\d+)\s*,\s*Recall\s*:\s*([\d.]+)\s*,\s*NDCG\s*:\s*([\d.]+)\s*,\s*Precision\s*([\d.]+)"
RESULT_REGEX = r"TODO_dien_regex_that"

m = re.search(RESULT_REGEX, log_text)
assert m, (
    f"Cell 6 đã chạy xong (log tồn tại tại {LOG_PATH}) nhưng không tìm thấy kết quả khớp RESULT_REGEX "
    "trong log — kiểm tra lại regex đã đúng định dạng dòng in kết quả thật của script gốc chưa, hoặc "
    "mở log lên xem script có báo lỗi gì trước đó không."
)

# TODO: map các group() của RESULT_REGEX sang tên cột kết quả thật, ví dụ (mẫu từ Phương án 1):
# best_epoch, recall, ndcg, precision = m.groups()
# result_df = pd.DataFrame([{
#     "Dataset": DATASET_NAME,
#     "Phương án": "<Tên Phương án>",
#     "Best Epoch": int(best_epoch),
#     "Recall@20": float(recall),
#     "NDCG@20": float(ndcg),
#     "Precision@20": float(precision),
#     "NUM_EPOCHS": NUM_EPOCHS,
#     # TODO (tuỳ chọn): thêm các hyperparameter riêng của phương án vào đây để PDF tự ghi chú lại cấu
#     # hình đã dùng, ví dụ "ANCHOR_W": ANCHOR_W, "SIGMA_MIN": SIGMA_MIN — hữu ích khi so sánh nhiều lần
#     # chạy khác nhau (mỗi lần chạy 1 file PDF riêng, xem cách đặt tên PDF_PATH bên dưới).
# }])
result_df = pd.DataFrame([{"Dataset": DATASET_NAME, "Phương án": "<Tên Phương án>", "raw_match": m.group(0)}])  # TODO: thay bằng bảng thật

display(result_df)
print()
print(result_df.to_markdown(index=False))

# ------------------------------------------------------------------
# Bảng chi tiết + biểu đồ biến thiên Recall/NDCG/Precision qua TỪNG epoch đã test (không chỉ epoch tốt
# nhất) — hạ tầng dùng chung, không cần sửa nếu Main.py giữ nguyên định dạng makePrint gốc của DiffMM.
# Nếu EPOCH_REGEX không khớp dòng nào, tự bỏ qua phần này (không báo lỗi, không chặn phần PDF bảng).
# ------------------------------------------------------------------
EPOCH_REGEX = r"Epoch\s+(\d+)/\d+,\s+Test:\s+Recall\s*=\s*([\d.]+),\s+NDCG\s*=\s*([\d.]+),\s+Precision\s*=\s*([\d.]+)"  # TODO: sửa nếu Main.py không dùng makePrint gốc

epochs, recalls, ndcgs, precisions = [], [], [], []
for line in log_text.split("\n"):
    m_line = re.search(EPOCH_REGEX, line)
    if m_line:
        ep_num, r_val, n_val, p_val = m_line.groups()
        epochs.append(int(ep_num))
        recalls.append(float(r_val))
        ndcgs.append(float(n_val))
        precisions.append(float(p_val))

if epochs:
    all_epochs_df = pd.DataFrame({
        "Epoch": epochs,
        "Recall@20": [f"{v:.5f}" for v in recalls],
        "NDCG@20": [f"{v:.5f}" for v in ndcgs],
        "Precision@20": [f"{v:.5f}" for v in precisions],
    })
    print("\n=== BẢNG CHỈ SỐ CHI TIẾT QUA TẤT CẢ EPOCH ĐÃ TEST ===")
    display(all_epochs_df)
else:
    print("\n(Không khớp được dòng log nào theo EPOCH_REGEX — bỏ qua bảng chi tiết + biểu đồ theo epoch. "
          "Sửa EPOCH_REGEX ở trên nếu Main.py in kết quả test theo định dạng khác.)")

# ------------------------------------------------------------------
# Xuất bảng kết quả cao nhất + tên dataset ra file PDF, rồi tự động tải về máy — KHÔNG cần sửa gì
# bên dưới, phần này dùng chung cho mọi phương án (tự lấy dữ liệu từ result_df/DATASET_NAME ở trên).
# ------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime


def _export_result_pdf(df, title_lines, out_path):
    n_rows, n_cols = len(df), len(df.columns)
    fig_w = min(max(6, 1.8 * n_cols), 18)
    fig_h = 1.6 + 0.55 * (n_rows + 1)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    ax.set_title("\n".join(title_lines), fontsize=12, fontweight="bold", loc="left", pad=14)
    tbl = ax.table(
        cellText=df.astype(str).values,
        colLabels=df.columns,
        cellLoc="center",
        loc="upper center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.auto_set_column_width(col=list(range(n_cols)))
    tbl.scale(1, 1.7)
    fig.tight_layout()
    fig.savefig(out_path, format="pdf", bbox_inches="tight")
    plt.close(fig)


_phuong_an_name = result_df["Phương án"].iloc[0] if "Phương án" in result_df.columns else "DiffMM"
_dataset_name = DATASET_NAME if "DATASET_NAME" in globals() and DATASET_NAME else (
    result_df["Dataset"].iloc[0] if "Dataset" in result_df.columns else "unknown_dataset"
)
_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

PDF_PATH = os.path.abspath(f"KetQua_{_dataset_name}.pdf")
_export_result_pdf(
    result_df,
    title_lines=[
        f"Kết quả huấn luyện — {_phuong_an_name}",
        f"Dataset: {_dataset_name}    |    Xuất lúc: {_timestamp}",
        # TODO (tuỳ chọn): thêm 1 dòng tóm tắt cấu hình đã dùng, ví dụ:
        # f"Configs: Epochs={NUM_EPOCHS}, AnchorW={ANCHOR_W}, SigmaMin={SIGMA_MIN}",
    ],
    out_path=PDF_PATH,
)
print(f"\nĐã xuất file PDF kết quả: {PDF_PATH}")

try:
    from google.colab import files as _colab_files
    _colab_files.download(PDF_PATH)
    print("Đã tự động tải file PDF về máy (kiểm tra thư mục Downloads của trình duyệt).")
except ImportError:
    print("Không chạy trong Colab nên không tự tải xuống — file PDF vẫn đã được lưu ở đường dẫn trên.")

# Ve bieu do 3-panel Recall/NDCG/Precision theo epoch (chi khi co du lieu tu EPOCH_REGEX o tren)
if epochs:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    _panels = [
        (recalls, "Recall@20", "#3b82f6", "o"),
        (ndcgs, "NDCG@20", "#f59e0b", "s"),
        (precisions, "Precision@20", "#10b981", "^"),
    ]
    for ax, (values, label, color, marker) in zip(axes, _panels):
        ax.plot(epochs, values, marker=marker, color=color, linewidth=2, label=label)
        ax.set_title(f"{label} over Epochs", fontsize=12, fontweight="bold", pad=10)
        ax.set_xlabel("Epoch", fontsize=10)
        ax.set_ylabel(label.split("@")[0], fontsize=10)
        ax.set_xlim(0, NUM_EPOCHS)
        ax.set_xticks(range(0, NUM_EPOCHS + 1, max(1, NUM_EPOCHS // 10)))
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.5f"))
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend(loc="lower right")

    fig.suptitle(f"Biến thiên chỉ số huấn luyện — {_dataset_name.upper()} ({_phuong_an_name})",
                 fontsize=14, fontweight="bold", y=1.05)
    fig.tight_layout()

    CHART_PDF_PATH = os.path.abspath(f"BieuDo_{_dataset_name}.pdf")
    fig.savefig(CHART_PDF_PATH, format="pdf", bbox_inches="tight")
    plt.show()
    print(f"Đã xuất biểu đồ: {CHART_PDF_PATH}")

    try:
        from google.colab import files as _colab_files
        _colab_files.download(CHART_PDF_PATH)
        print("Đã tự động tải file PDF biểu đồ về máy.")
    except ImportError:
        pass


## 8. Cell tối ưu siêu tham số (TÙY CHỌN — Optuna)

Cell này **không bắt buộc** cho quy trình bàn giao chuẩn — chỉ chạy nếu muốn tự động dò bộ siêu tham số
tốt nhất thay vì quét thủ công. Dùng [Optuna](https://optuna.org/) (Bayesian Optimization, TPE sampler)
để chạy nhiều lượt huấn luyện ngắn (`OPTUNA_EPOCHS` mỗi lượt, ít hơn `NUM_EPOCHS` đầy đủ), mỗi lượt thử
1 bộ tham số khác nhau, rồi báo bộ tham số cho `Recall@20` cao nhất. **Yêu cầu chạy xong Cell 1→5 trước**
(cần `REPO_DIR`, `MAIN_SCRIPT_NAME`, dữ liệu đã tải sẵn).

**Cần điền TODO:** khai báo `trial.suggest_...` cho ĐÚNG các hyperparameter riêng của phương án (tên +
khoảng giá trị hợp lý), và đảm bảo `CLI_ARGS` bên trong `objective()` khớp với `Params.py` thật của bản
fork — không đoán tên tham số.


In [ ]:
# ============================================================
# CELL 8 (TÙY CHỌN) — TỐI ƯU HÓA SIÊU THAM SỐ BẰNG OPTUNA
# ============================================================
import os
import re
import subprocess

try:
    import optuna
except ImportError:
    print("Đang cài đặt Optuna...")
    subprocess.run(["pip", "install", "-q", "optuna"])
    import optuna

N_TRIALS = 10        # TODO: số lượt thử nghiệm — tăng lên 20-30 nếu có nhiều thời gian
OPTUNA_EPOCHS = 25   # TODO: số epoch chạy thử mỗi lượt — đủ lớn để mô hình kịp hội tụ, nhưng nhỏ hơn NUM_EPOCHS đầy đủ để tiết kiệm thời gian


def objective(trial):
    # TODO: điền đúng tên + khoảng giá trị hyperparameter riêng của phương án, ví dụ (mẫu từ Phương án 6/7):
    # opt_anchor_w = trial.suggest_float("anchor_w", 0.5, 5.0)
    # opt_sigma_min = trial.suggest_float("sigma_min", 1e-4, 5e-3, log=True)
    raise NotImplementedError("TODO: điền trial.suggest_... + CLI_ARGS cho phương án này rồi xoá dòng raise này.")

    # TODO: build CLI_ARGS y hệt Cell 6, nhưng dùng OPTUNA_EPOCHS và các giá trị trial.suggest_ ở trên
    # DATASET_HP = {...}  # giống Cell 6
    # CLI_ARGS = ["--data", DATASET_NAME, "--epoch", str(OPTUNA_EPOCHS), ...] + DATASET_HP[DATASET_NAME]

    log_path = os.path.abspath("train_log.txt")
    if os.path.exists(log_path):
        os.remove(log_path)  # xoa log cu truoc moi luot, tranh doc nham ket qua luot truoc

    cmd = ["python", MAIN_SCRIPT_NAME] + CLI_ARGS
    with open(log_path, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=log_file, stderr=subprocess.STDOUT)
        process.wait()

    if not os.path.exists(log_path):
        return 0.0
    with open(log_path, "r", encoding="utf-8") as f:
        log_text = f.read()

    # TODO: dung dung RESULT_REGEX da xac nhan o Cell 7
    m = re.search(RESULT_REGEX, log_text)
    if m:
        best_recall = float(m.groups()[1])  # TODO: doi chi so group() cho dung cot Recall trong RESULT_REGEX
        print(f"-> Lượt {trial.number} kết thúc. Recall@20 đạt: {best_recall:.5f}")
        return best_recall
    print(f"-> Lượt {trial.number} thất bại hoặc không ghi nhận kết quả. Đánh giá: 0.0")
    return 0.0


print("Bắt đầu tối ưu hóa bằng Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print("\n" + "=" * 40)
print("TỐI ƯU HÓA HOÀN TẤT!")
print("Bộ tham số tốt nhất:", study.best_params)
print(f"Recall@20 tốt nhất đạt: {study.best_value:.5f}")
print("=> Quay lại Cell 1, điền các giá trị tối ưu này vào biến tương ứng, đặt lại NUM_EPOCHS đầy đủ, "
      "rồi chạy lại Cell 6 & 7 để huấn luyện + xuất kết quả cuối cùng.")
